# 02 — Slippage Proxy Construction and Alpha Calibration

## Proxy formula
```
arrival_price     = close_t
base_exec_price   = (open_{t+1} + high_{t+1} + low_{t+1} + close_{t+1}) / 4
impact_penalty    = α * order_size_fraction * vol_rolling * arrival_price
exec_price        = base_exec_price + side * impact_penalty
slippage          = side * (exec_price - arrival_price)
slippage_bps      = 10_000 * slippage / arrival_price
```

## Alpha calibration
α is swept over `[0.5, 1.0, 2.0, 5.0, 10.0]`. For each candidate we build
the proxy and compare label distributions on the validation set. The α that
minimises MAE vs the reference proxy (α=2.0) is selected and documented.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from slippage.data_loader import TICKERS, download_ohlcv
from slippage.features import compute_market_features, add_synthetic_orders
from slippage.proxy import build_proxy, calibrate_alpha
from slippage.dataset import temporal_split
from slippage.pipeline import build_full_proxy, _ticker_seed
from slippage.viz import plot_alpha_sensitivity, FIGURES_DIR

In [ ]:
data = download_ohlcv(TICKERS)
proxy_all = build_full_proxy(data, alpha=2.0)
print(f'Total proxy rows: {len(proxy_all):,}')
print(proxy_all[['slippage_bps', 'order_size_fraction', 'vol_rolling', 'side']].describe())

In [ ]:
# Slippage distribution
fig, ax = plt.subplots(figsize=(8, 4))
proxy_all['slippage_bps'].clip(-200, 200).hist(bins=100, ax=ax, color='steelblue', edgecolor='none')
ax.set_xlabel('Slippage (bps)')
ax.set_title('Synthetic Slippage Distribution (all tickers, clipped ±200 bps)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'slippage_distribution.png', dpi=150)
plt.show()

In [ ]:
# Alpha calibration on SPY validation split.
# Use iloc-based slicing on the proxy (which already contains the order
# features). Avoids loc-based duplicate-index issues entirely.
spy_df = data['SPY']
spy_mf = compute_market_features(spy_df)
spy_rng = np.random.default_rng(_ticker_seed('SPY'))
spy_feats = add_synthetic_orders(spy_mf, rng=spy_rng)
spy_proxy_ref = build_proxy(spy_df, spy_feats, alpha=2.0)

# Chronological val slice (65–80% of bars), aligned row-by-row.
n = len(spy_proxy_ref)
i_val, i_test = int(0.65 * n), int(0.80 * n)
val_feats = spy_proxy_ref.iloc[i_val:i_test]
val_target = val_feats['slippage_bps'].values

alphas = [0.5, 1.0, 2.0, 5.0, 10.0]
best_alpha, sensitivity = calibrate_alpha(spy_df, val_feats, val_target, alphas=alphas)
print(f'Alpha sensitivity: {sensitivity}')
print(f'Best alpha: {best_alpha}')

In [ ]:
fig = plot_alpha_sensitivity(
    alphas=[a for a in alphas if a in sensitivity],
    val_maes=[sensitivity[a] for a in alphas if a in sensitivity],
    best_alpha=best_alpha,
    save_as='alpha_sensitivity.png',
)
plt.show()
print(f'Selected alpha = {best_alpha}  (minimises MAE on validation set)')